### Tools & Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

### Load Data

In [2]:
data = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Exploratory Data Analysis (EDA)

In [3]:
data.info()  # Check missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
data.describe()  # Summary statistics

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


### Feature Engineering

**1. Family Features**

In [5]:
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
data['IsAlone'] = np.where(data['FamilySize'] == 1, 1, 0)

**2. Title Extraction from Name**

In [6]:
data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
# Simplify rare titles
data['Title'] = data['Title'].replace(['Lady', 'Countess','Capt', 'Col', 
                                       'Don', 'Dr', 'Major', 'Rev', 'Sir', 
                                       'Jonkheer', 'Dona'], 'Rare')
data['Title'] = data['Title'].replace('Mlle', 'Miss')
data['Title'] = data['Title'].replace('Ms', 'Miss')
data['Title'] = data['Title'].replace('Mme', 'Mrs')

**3. Age Binning**

In [7]:
data['AgeBin'] = pd.cut(data['Age'], bins=[0, 12, 20, 40, 60, 80], labels=[0,1,2,3,4])

**4. Fare Binning**

In [8]:
data['FareBin'] = pd.qcut(data['Fare'], 4, labels=[0,1,2,3])

**5. Cabin Feature**

In [9]:
data['CabinLetter'] = data['Cabin'].str[0]  # Take first letter
data['CabinLetter'] = data['CabinLetter'].fillna('U')  # U = Unknown

### Handle Missing Values

In [10]:
# Age: fill with median per Title
data['Age'] = data.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

# Embarked: fill with mode
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

# Fare: fill missing values with median
data['Fare'] = data['Fare'].fillna(data['Fare'].median())

### Encode Categorical Features

In [11]:
categorical_features = ['Sex', 'Embarked', 'Title', 'CabinLetter']
numeric_features = ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch', 'IsAlone']

In [12]:
# Column transformer
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features)
])

### Train-Test Split

In [13]:
X = data.drop(['Survived', 'Name', 'Ticket', 'Cabin'], axis=1)
y = data['Survived']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Model Training

In [15]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))
])

In [17]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

0.7988826815642458

### Feature Importance

In [18]:
# Get feature names after one-hot encoding
ohe_features = model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_features = numeric_features + list(ohe_features)

In [19]:
# Feature importance
importances = model.named_steps['classifier'].feature_importances_
feat_importance = pd.DataFrame({'Feature': all_features, 'Importance': importances}).sort_values(by='Importance', ascending=False)
feat_importance

,Feature,Importance
1,Fare,0.242797
0,Age,0.205393
6,Sex_male,0.118594
10,Title_Mr,0.117923
2,FamilySize,0.053637
11,Title_Mrs,0.040542
3,SibSp,0.039761
20,CabinLetter_U,0.035322
9,Title_Miss,0.031330
4,Parch,0.025078


#### Summary of Features Engineered

- FamilySize, IsAlone – capture family dynamics
- Title – derived from Name
- AgeBin – age categories
- FareBin – fare categories
- CabinLetter – first letter of Cabin (deck info)
- SibSp, Parch – original family features
- Scaled numerical features and one-hot encoded categorical features

**This is full A→Z feature engineering, ready for ML models.**